**21 | HARTH Minority-Class and Source Data Quality Audit**

**Human Movement Intelligence | Presented by: Reza Saadatyar (2026)**

**Aim:** Determine how source-label prevalence, activity-bout duration, temporal discontinuities, deterministic window caps and participant differences affect the *available evidence* for the eight-class HARTH task. Focus on Shuffling and stair activities without assuming that data characteristics prove the cause of prediction errors.

**Evidence boundaries:** Official pinned HARTH v2.0 source CSVs, 29 development participants only. S023/S032 remain excluded, although inspected in early historical stages. Source codes without documented meanings retain their numeric identities. No model, scaler or classifier is fitted here. All results are genuine, generated locally when this notebook is executed.

**01. Research Questions and Quality-Audit Design**

We address five distinct questions: (1) Which task activities are rare by *windows* versus independent *participants*? (2) Do short annotated activity bouts limit the number of pure two-second windows? (3) How many candidate windows are discarded by activity boundaries, timestamp gaps or both? (4) Do undocumented codes occur in raw recordings and what exclusion follows strict windowing? (5) Do the six raw accelerometer axes show apparent missingness, long repeated sequences or unusual ranges requiring follow-up?

**Important:** These are descriptive diagnostics rather than proof of defective source annotations, physical sensor malfunction or reasons for any specific classifier error.

In [ ]:
# ==================== 01. Study Environment and Fixed Protocol ====================

# 1. Resolve the repository root and reuse the documented research modules.
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
root = Path.cwd().resolve()
while not (root / "code/python/src/core/data_loading.py").is_file():
    if root.parent == root:
        raise RuntimeError("Run inside human-movement-intelligence.")
    root = root.parent
sys.path.insert(0, str(root / "code/python/src"))
sys.path.insert(0, str(root / "studies/wearable_activity_recognition/src"))

# 2. Import the official study identifiers and source-quality audit functions.
from harth_data import HARTH_SUBJECTS, ACTIVITY_NAMES, SENSOR_COLUMNS, HARTH_REVISION
from harth_quality import audit_harth_development_sources
from core.sampling_audit import audit_temporal_sampling

# 3. Freeze scientific classification and protected Test boundaries.
RESERVED_TEST = {"S023", "S032"}
SUBJECTS = sorted(set(HARTH_SUBJECTS) - RESERVED_TEST)
CLASSES = tuple(range(1, 9))
DATA_DIR = root / "data/raw/harth_v2"
assert len(SUBJECTS) == 29
assert not (set(SUBJECTS) & RESERVED_TEST)

print("\n" + "=" * 65)
print("STAGE 21 | HARTH SOURCE QUALITY AUDIT")
print("=" * 65)
print("Development subjects:", len(SUBJECTS))
print("Documented target activity codes:", CLASSES)
print("Pinned source revision:", HARTH_REVISION)
print("Reserved Test subjects excluded:", sorted(RESERVED_TEST))
print("=" * 65)

**02. Read and Independently Audit Each Development Recording**

The audit loads each person's pinned public CSV separately, preserves raw annotation codes and channels, checks 50 Hz timestamp continuity, forms 100-sample windows with 50-sample stride, and accepts only time-continuous, perfectly single-labelled candidates. Unknown and cycling codes are removed from the eight-class cohort **only after** valid source windows are formed.

For each participant it records original sample counts, mutually exclusive window rejection reasons, accepted task windows, source annotation segments and sensor-axis summaries. This is more expensive than reading an already-prepared summary but avoids inferring new data-quality findings from older plots.

In [ ]:
# ==================== 02. Genuine Source-Data Quality Audit ====================

# 1. Process each person individually; never access protected source IDs.
(overview, raw_codes, window_codes, metadata,
 bouts, channels, examples) = audit_harth_development_sources(
    subjects=SUBJECTS, data_dir=DATA_DIR, task_classes=CLASSES,
    window_size=100, step_size=50, sampling_rate=50,
    reserved_subjects=RESERVED_TEST,
)

# 2. Display a transparent cohort accounting table before any interpretation.
display(overview.sort_values("subject_id"))
print("\n" + "=" * 65)
print("DEVELOPMENT SOURCE ACCOUNTING")
print("=" * 65)
print("Source participants:", overview["subject_id"].nunique())
print("Original CSV rows:", f'{overview["raw_samples"].sum():,}')
print("All valid pure-label windows:", f'{overview["retained_pure_time_safe"].sum():,}')
print("Eight-class eligible windows:", f'{len(metadata):,}')
print("Windows excluded after segmentation:", f'{overview["excluded_after_windowing"].sum():,}')
print("=" * 65)

# 3. Verify historical eight-class cohort size without silently reclassifying data.
assert len(metadata) == 132397, "Investigate source/pipeline mismatch with Stage 08."
assert set(metadata["subject_id"]) == set(SUBJECTS)

**03. Raw Source Labels: Documented, Out-of-Task and Unmapped Codes**

Distinguish the eight selected target classes from other officially documented activities (for example cycling codes) and *undocumented* codes. Both categories are excluded from the target task, but they are not interchangeable. Report the original row support and accepted post-segmentation window support by code and participant. Do not give descriptive names to codes `10`, `11` or `17` without official source confirmation.

In [ ]:
# ==================== 03. Source Label Provenance and Exclusions ====================

# 1. Report observed original label-code frequencies, preserving raw numbers.
source_labels = raw_codes.groupby(
    ["source_code", "source_activity", "in_eight_class_task"],
    as_index=False
)["source_samples"].sum().sort_values("source_code")
display(source_labels)

# 2. Audit all annotation codes that cannot enter the eight-class task.
excluded_codes = raw_codes.loc[
    ~raw_codes["in_eight_class_task"]
].sort_values(["source_code", "subject_id"])
display(excluded_codes[[
    "subject_id", "source_code", "source_activity",
    "source_samples", "documented_source_code"
]])

# 3. Show precisely how many accepted pure windows were excluded by their code.
post_window = window_codes.groupby(
    ["source_code", "source_activity", "in_eight_class_task"],
    as_index=False
)["accepted_pure_windows"].sum().sort_values("source_code")
display(post_window)
unknown_codes = sorted(
    raw_codes.loc[~raw_codes["documented_source_code"], "source_code"].unique().tolist()
)
print("Undocumented source codes observed:", unknown_codes)
print("Unknown-code semantic interpretation performed: NO")

**04. Activity Prevalence and Independent Participant Support**

Class frequency in overlapping windows measures *labelled window volume*, not independent observations. A class can have many windows concentrated in a few participants; comparing participant support and within-subject fractions exposes that distinction. Missing classes are represented as zero rather than disappearing from tables.

In [ ]:
# ==================== 04. Participant-Level Activity Coverage ====================

# 1. Build the complete participant-by-class count matrix with explicit zeros.
coverage = pd.crosstab(metadata["subject_id"], metadata["label"]).reindex(
    index=SUBJECTS, columns=CLASSES, fill_value=0
).astype(int)
coverage.columns = [ACTIVITY_NAMES[c] for c in CLASSES]
display(coverage)

# 2. Quantify both sample prevalence and the number of distinct contributors.
class_summary = pd.DataFrame({
    "activity": coverage.columns,
    "eligible_windows": coverage.sum(axis=0).to_numpy(),
    "supporting_subjects": (coverage > 0).sum(axis=0).to_numpy(),
})
class_summary["window_share_pct"] = (
    class_summary["eligible_windows"] / class_summary["eligible_windows"].sum() * 100
)
class_summary["subject_coverage_pct"] = (
    class_summary["supporting_subjects"] / len(SUBJECTS) * 100
)
display(class_summary.round(2))

# 3. Visualise activity volume and participant support as separate quantities.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].barh(class_summary["activity"], class_summary["eligible_windows"])
axes[0].set(title="Eligible windows per activity", xlabel="Overlapping windows")
axes[1].barh(class_summary["activity"], class_summary["supporting_subjects"])
axes[1].set(title="Independent participant support", xlabel="Participants", xlim=(0,29))
for ax in axes:
    ax.grid(axis="x", alpha=0.2)
fig.tight_layout()
plt.show()

**05. Participant-by-Class Heatmap and Within-Person Composition**

Two complementary views are required: absolute class counts show observational volume, while percentages show how a participant's available activity mix differs. Neither view should be interpreted as subject prevalence in the general population. Rare classes must remain visible even when their values are small.

In [ ]:
# ==================== 05. Participant and Activity Distribution Maps ====================

# 1. Preserve class order while comparing raw counts and within-person fractions.
fractions = coverage.div(coverage.sum(axis=1), axis=0).mul(100)
fig, axes = plt.subplots(1, 2, figsize=(16, 10), sharey=True)
maps = [
    (np.log1p(coverage.to_numpy()), "log(1 + windows)", "Window counts (log scale)"),
    (fractions.to_numpy(), "Percent of participant windows", "Within-person activity (%)"),
]

# 2. Plot separate heatmaps with identical participant/class positions.
for ax, (matrix, unit, title) in zip(axes, maps):
    handle = ax.imshow(matrix, aspect="auto", interpolation="nearest")
    ax.set_xticks(range(len(CLASSES)), coverage.columns, rotation=70, ha="right")
    ax.set_yticks(range(len(SUBJECTS)), SUBJECTS)
    ax.set(title=title, xlabel="Documented activity")
    fig.colorbar(handle, ax=ax, label=unit, fraction=0.046, pad=0.02)
axes[0].set_ylabel("Development participant")
fig.tight_layout()
plt.show()

**06. Annotation-Bout Duration and Short-Activity Evidence**

An annotation bout is a run of identical source labels separated by label changes or source timestamp discontinuities. Because the model uses two-second windows with strict label purity, bouts shorter than 100 consecutive samples cannot contain a complete candidate window; longer bouts may also be missed depending on stride alignment. These statistics describe source annotations, not ground-truth behavioural event lengths.

In [ ]:
# ==================== 06. Minority-Class Annotation Bout Analysis ====================

# 1. Restrict bout analysis to documented task labels, without reinterpreting others.
task_bouts = bouts.loc[bouts["source_code"].isin(CLASSES)].copy()
focus = task_bouts.loc[task_bouts["source_code"].isin([3, 4, 5])].copy()
bout_summary = task_bouts.groupby("source_code").agg(
    bouts=("samples", "size"),
    median_duration_s=("nominal_duration_seconds", "median"),
    mean_duration_s=("nominal_duration_seconds", "mean"),
    short_bouts=("shorter_than_window", "sum"),
    contributors=("subject_id", "nunique"),
).reindex(CLASSES)
bout_summary["activity"] = [ACTIVITY_NAMES[c] for c in CLASSES]
bout_summary["short_bout_pct"] = bout_summary["short_bouts"] / bout_summary["bouts"] * 100
display(bout_summary.reset_index()[[
    "source_code", "activity", "bouts", "contributors",
    "median_duration_s", "mean_duration_s", "short_bouts", "short_bout_pct"
]].round(3))

# 2. Plot the distribution of recorded minority-activity bout durations.
fig, ax = plt.subplots(figsize=(11, 5))
values = [
    focus.loc[focus["source_code"] == c, "nominal_duration_seconds"]
    for c in [3, 4, 5]
]
ax.boxplot(values, tick_labels=[ACTIVITY_NAMES[c] for c in [3, 4, 5]],
           showfliers=False)
ax.axhline(2, linestyle="--", color="gray", label="100 samples at 50 Hz")
ax.set(title="Minority activity annotation-bout durations",
       ylabel="Nominal duration (seconds)")
ax.legend()
ax.grid(axis="y", alpha=0.2)
fig.tight_layout()
plt.show()

**07. Candidate-Window Rejection: Activity Boundaries versus Timestamp Gaps**

Candidate windows can be rejected for pure-label violations, time discontinuities or both. We count these categories as mutually exclusive; adding separate transition and gap totals without separating overlap would exaggerate the total number of rejected candidates. Timestamp anomalies are audit signals, not automatically proven measurement corruption.

In [ ]:
# ==================== 07. Window Rejection and Timestamp Integrity ====================

# 1. Summarise mutually exclusive candidate-window decisions.
reasons = [
    "retained_pure_time_safe", "rejected_label_only",
    "rejected_time_only", "rejected_both"
]
reason_summary = pd.DataFrame({
    "decision": reasons,
    "windows": [int(overview[name].sum()) for name in reasons]
})
reason_summary["share_pct"] = (
    reason_summary["windows"] / overview["candidate_windows"].sum() * 100
)
display(reason_summary.round(2))

# 2. Show which source participants contribute most rejected candidates.
per_subject_rejection = overview[[
    "subject_id", "candidate_windows",
    "rejected_label_only", "rejected_time_only", "rejected_both",
    "nonpositive_intervals", "long_intervals",
    "median_positive_interval_seconds"
]].copy()
per_subject_rejection["rejection_pct"] = (
    (per_subject_rejection[["rejected_label_only",
      "rejected_time_only", "rejected_both"]].sum(axis=1))
    / per_subject_rejection["candidate_windows"] * 100
)
display(per_subject_rejection.sort_values("rejection_pct", ascending=False).round(3))

# 3. Visualise raw rejection categories without conflating shared boundaries.
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(reason_summary["decision"], reason_summary["windows"])
ax.set(title="Mutually exclusive source-window decisions", ylabel="Candidate windows")
ax.tick_params(axis="x", rotation=20)
ax.grid(axis="y", alpha=0.2)
fig.tight_layout()
plt.show()

**08. Sampling-Cap Sensitivity for Minority Classes**

Reproduce chronological, label-blind caps of 250, 500 and 1,000 against the complete eligible cohort. The separate cap-1,000 policy here is NOT the strictly nested training-expansion policy used in later modelling stages. Reporting both class retention and people still represented prevents conflating class volume with participant coverage.

In [ ]:
# ==================== 08. Chronological Cap and Class Retention Audit ====================

# 1. Reuse exact post-segmentation eligible source rows from this stage.
selected, per_subject, cap_counts = audit_temporal_sampling(
    metadata=metadata, caps=(250, 500, 1000),
    classes=CLASSES, forbidden_subjects=RESERVED_TEST,
)
cap_order = ["full", "cap_250", "cap_500", "cap_1000"]
count_table = cap_counts.pivot(
    index="label", columns="policy", values="windows"
).reindex(index=CLASSES, columns=cap_order).astype(int)
retention = count_table.div(count_table["full"], axis=0) * 100
retention.index = [ACTIVITY_NAMES[c] for c in CLASSES]
display(count_table)
display(retention.round(2))

# 2. Count how many genuinely distinct people remain per class under each cap.
support_table = selected.groupby(["label", "policy"])["subject_id"].nunique().unstack(
    fill_value=0
).reindex(index=CLASSES, columns=cap_order, fill_value=0).astype(int)
support_table.index = [ACTIVITY_NAMES[c] for c in CLASSES]
display(support_table)

# 3. Plot rare-class retention and contributor support in one two-panel figure.
minority_names = [ACTIVITY_NAMES[c] for c in (3, 4, 5)]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
retention.loc[minority_names, ["cap_250","cap_500","cap_1000"]].plot.bar(
    ax=axes[0], rot=15, title="Minority-class window retention (%)"
)
support_table.loc[minority_names, ["cap_250","cap_500","cap_1000"]].plot.bar(
    ax=axes[1], rot=15, title="Independent participants retained"
)
axes[0].set(ylabel="Percent of full eligible windows", ylim=(0, 105))
axes[1].set(ylabel="Participants", ylim=(0, 29))
for ax in axes:
    ax.grid(axis="y", alpha=0.2)
fig.tight_layout()
plt.show()

**09. Raw Accelerometer Channel Diagnostics and Example Signals**

Describe distributions for the original six accelerometer axes without applying a learned scaler. The source metadata must be consulted before attaching physical units or making instrumentation claims. In particular, high values, tiny SD, or repeated adjacent numbers warrant inspection but do not independently prove clipping or sensor failure.

Show genuine two-second examples for Walking, Shuffling and both stair activities, each with an explicit original participant and source window start. These are illustrative only and not proof of a systematic morphological difference.

In [ ]:
# ==================== 09. Raw Signal Distribution and Source Examples ====================

# 1. Audit numeric signal ranges and repeated adjacent values by channel.
channel_summary = channels.groupby("channel").agg(
    recordings=("subject_id", "nunique"),
    median_std=("std", "median"),
    median_absolute_p99=("absolute_p99", "median"),
    largest_absolute_max=("maximum", "max"),
    smallest_minimum=("minimum", "min"),
    mean_repeated_adjacent_pct=("repeated_adjacent_pct", "mean"),
    nonfinite_samples=("nonfinite_samples", "sum"),
)
display(channel_summary.round(4))
display(channels.sort_values("repeated_adjacent_pct", ascending=False).head(12).round(4))

# 2. Display actual window examples without inferred physiological annotations.
chosen_codes = [1, 3, 4, 5]
fig, axes = plt.subplots(len(chosen_codes), 1, figsize=(13, 12), sharex=True)
for ax, code in zip(axes, chosen_codes):
    item = examples.get(code)
    if item is None:
        ax.text(.5, .5, "No eligible source window", ha="center", va="center")
        continue
    for index, name in enumerate(item["channel_names"]):
        ax.plot(np.arange(item["signals"].shape[1]) / 50,
                item["signals"][index], label=name, linewidth=1)
    ax.set_title(f'{ACTIVITY_NAMES[code]} | {item["subject_id"]} | '
                 f'source index {item["start_index"]}')
    ax.set_ylabel("Raw source value")
    ax.grid(alpha=0.15)
axes[0].legend(ncol=3, fontsize=8)
axes[-1].set_xlabel("Time within source window (seconds)")
fig.tight_layout()
plt.show()

**10. Interpret Model-Error Context Without Causal Overclaiming**

Earlier model experiments found low Shuffling recall, especially for Standard CNN and Weighted CNN. Those previously saved results are used here only as *context*: Stage 13 reported pooled out-of-fold Shuffling F1 of 0.4957 for Random Forest, 0.0000 for Standard CNN and 0.1345 for Weighted CNN. Stage 18 reported 27 correctly labelled Shuffling windows out of 219 for its own 400-step CNN protocol.

Those experiments differ in training controls, so avoid treating them as interchangeable rows in a leaderboard. Source-data properties measured in this notebook can motivate hypotheses, but testing causal model effects requires Stage 22–24 controlled experiments.

In [ ]:
# ==================== 10. Minority-Class Evidence and Future Hypotheses ====================

# 1. Document source-backed context, not new predictions from this stage.
historical = pd.DataFrame([
    {"experiment": "Stage 13 (matched 3-fold)", "model": "Random Forest",
     "shuffling_f1": 0.4957, "protocol_note": "500-cap model comparison"},
    {"experiment": "Stage 13 (matched 3-fold)", "model": "Standard CNN",
     "shuffling_f1": 0.0000, "protocol_note": "500-cap model comparison"},
    {"experiment": "Stage 13 (matched 3-fold)", "model": "Weighted CNN",
     "shuffling_f1": 0.1345, "protocol_note": "500-cap model comparison"},
])
display(historical)

# 2. Separate observations from hypotheses to test later.
hypotheses = pd.DataFrame([
    {"observation": "Low retained Shuffling window volume",
     "possible_investigation": "Stratified within-Train sampling or loss ablation",
     "proof_of_cause": "Not established"},
    {"observation": "Uneven activity presence by participant",
     "possible_investigation": "Group-held-out class support and fold diagnostics",
     "proof_of_cause": "Not established"},
    {"observation": "Short annotation bouts and strict 2-second windows",
     "possible_investigation": "Sensitivity to justified window lengths",
     "proof_of_cause": "Not established"},
    {"observation": "Unknown source annotation codes",
     "possible_investigation": "Request clarification from official maintainers",
     "proof_of_cause": "Not established"},
])
display(hypotheses)
print("Models trained in Stage 21: NONE")
print("Prediction errors recomputed from raw signals: NO")

**11. Scientific Quality Gate and Hand-Off to Stage 22**

The final gate verifies participant identities, source-row and window-count reconciliation, source labels, sample retention and no protected Test access. These checks do not independently certify annotation correctness, instrument calibration, external generalisation or model superiority. Stage 22 will implement **all four** agreed temporal architectures: TCN, Multi-Scale CNN, 1D-ResNet and Time-Series Transformer. The advanced architectures will then be trained on a matched protocol in Stage 23.

In [ ]:
# ==================== 11. Research Quality Gate ====================

# 1. Confirm source identity, class scope and protected Test exclusion.
assert overview["subject_id"].nunique() == len(SUBJECTS) == 29
assert not (set(raw_codes["subject_id"]) & RESERVED_TEST)
assert not (set(metadata["subject_id"]) & RESERVED_TEST)
assert set(metadata["label"].unique()).issubset(CLASSES)

# 2. Confirm all candidate windows were assigned one exclusive decision.
decision_columns = [
    "retained_pure_time_safe", "rejected_label_only",
    "rejected_time_only", "rejected_both",
]
assert overview[decision_columns].sum(axis=1).eq(
    overview["candidate_windows"]
).all()
assert len(metadata) == int(overview["eligible_eight_class_windows"].sum()) == 132397

# 3. Confirm counts and retention remain consistent with source audit.
assert int(coverage.to_numpy().sum()) == len(metadata)
assert count_table["full"].sum() == len(metadata)
assert int(count_table["cap_500"].sum()) == 14201
assert (retention <= 100 + 1e-8).to_numpy().all()
assert channel_summary["nonfinite_samples"].eq(0).all()
assert len(examples) == len(CLASSES)

# 4. Print an interpretable evidence-bounded completion summary.
print("\n" + "=" * 65)
print("STAGE 21: MINORITY-CLASS AND DATA QUALITY ANALYSIS")
print("=" * 65)
print("Development source participants:", len(SUBJECTS))
print("Documented activity classes:", len(CLASSES))
print("Eligible 100-sample windows:", f"{len(metadata):,}")
print("Original Test subjects accessed: NO")
print("Source label preservation: PASSED")
print("Window-rejection reconciliation: PASSED")
print("Participant/class coverage audit: PASSED")
print("Sampling-cap consistency: PASSED")
print("Raw sensor descriptive checks: PASSED")
print("-" * 65)
print("Stage 21 scientific quality gate: PASSED")
print("Models trained or evaluated here: NO")
print("Independent generalisation established: NO")
print("Next stage: Stage 22 - Four Advanced Temporal Architectures")
print("=" * 65)